# Egypt & MENA Digital Financial Inclusion Opportunity Analytics

## 1. Notebook purpose
Prepare the World Bank Global Findex 2025 workbook for comparisons across seven MENA markets. The output retains technical indicator codes and demographic detail for traceability. The IMF supply-side dataset is prepared separately.

## 2. Imports and file paths
Use project-relative paths so the notebook runs from the `notebooks/` directory. The raw workbook remains unchanged.

In [1]:
import pandas as pd

file_path = "../data/raw/GlobalFindexDatabase2025.xlsx"
excel_file = pd.ExcelFile(file_path)
print(excel_file.sheet_names)

['Notes', 'Data', 'Series Table', 'Updates']


## 3. Raw workbook inspection
Inspect sheet names and the first unparsed rows before choosing the header and skipping the workbook’s second row.

In [2]:
data_preview = pd.read_excel(
    file_path,
    sheet_name="Data",
    header=None,
    nrows=10
)

data_preview.iloc[:, :12]

Out[0]: 
             0   ...                               11
0  countrynewwb  ...                   borrow.any.t.d
1       Economy  ...  Borrowed any money (%, age 15+)
2   Afghanistan  ...                              NaN
3       Albania  ...                              NaN
4       Algeria  ...                              NaN
5        Angola  ...                              NaN
6     Argentina  ...                              NaN
7       Armenia  ...                              NaN
8     Australia  ...                              NaN
9       Austria  ...                              NaN

[10 rows x 12 columns]


## 4. Loading and understanding the data
Inspect the parsed shape, columns, and demographic fields to establish the source structure.

In [3]:
data = pd.read_excel(
    file_path,
    sheet_name="Data",
    header=0,
    skiprows=[1]
)

data.head()

Out[0]: 
  countrynewwb codewb  year   pop_adult  ... con32g con32g.s con32h con32h.s
0  Afghanistan    AFG  2011  14575546.0  ...    NaN      NaN    NaN      NaN
1      Albania    ALB  2011   2281010.0  ...    NaN      NaN    NaN      NaN
2      Algeria    DZA  2011  26251587.0  ...    NaN      NaN    NaN      NaN
3       Angola    AGO  2011  12779501.0  ...    NaN      NaN    NaN      NaN
4    Argentina    ARG  2011  30685516.0  ...    NaN      NaN    NaN      NaN

[5 rows x 466 columns]


In [4]:
data.shape

Out[0]: (8577, 466)


In [5]:
data.columns[:20]

Out[0]: 
Index(['countrynewwb', 'codewb', 'year', 'pop_adult', 'regionwb24_hi',
       'incomegroupwb24', 'group', 'group2', 'account.t.d', 'fiaccount.t.d',
       'mobileaccount.t.d', 'borrow.any.t.d', 'fin4.d', 'fin11_2a',
       'fin11_2a.s', 'fin11a', 'fin11a.s', 'fin11b', 'fin11b.s', 'fin11c'],
      dtype='object')


## 5. Country selection
Restrict analysis to the seven project markets. Egypt retains its source label until final standardization.

In [6]:
core_countries = [
    "Egypt, Arab Rep.",
    "Morocco",
    "Tunisia",
    "Algeria",
    "Jordan"
]

optional_benchmarks = [
    "Saudi Arabia",
    "United Arab Emirates"
]

project_countries = core_countries + optional_benchmarks

In [7]:
project_data = data[
    data["countrynewwb"].isin(project_countries)
].copy()

In [8]:
print(project_data.shape)
print(project_data["countrynewwb"].unique())

(356, 466)
['Algeria' 'Egypt, Arab Rep.' 'Jordan' 'Saudi Arabia'
 'United Arab Emirates' 'Tunisia' 'Morocco']


In [9]:
print(project_data["group"].value_counts())

group
gender        62
income        62
age_cat       62
education     62
laborforce    62
all           32
urbanicity    14
Name: count, dtype: int64


In [10]:
print(project_data["group2"].value_counts())

group2
all                      32
men                      31
women                    31
richest 60%              31
poorest 40%              31
age 25+                  31
ages 15-24               31
in laborforce            31
out of laborforce        31
secondary edu or more    31
prim edu or less         31
urban                     7
rural                     7
Name: count, dtype: int64


## 6. Year and demographic coverage
Check actual observations by year and country; availability varies across waves and demographics.

In [11]:
print(project_data["year"].value_counts().sort_index())

year
2011    45
2014    66
2017    77
2021    77
2024    91
Name: count, dtype: int64


In [12]:
country_year_coverage = (
    project_data[project_data["group"] == "all"]
    .pivot_table(
        index="countrynewwb",
        columns="year",
        values="codewb",
        aggfunc="count",
        fill_value=0
    )
)

country_year_coverage

Out[0]: 
year                  2011  2014  2017  2021  2024
countrynewwb                                      
Algeria                  1     1     1     1     1
Egypt, Arab Rep.         1     1     1     1     1
Jordan                   1     1     1     1     1
Morocco                  0     0     1     1     1
Saudi Arabia             1     1     1     1     1
Tunisia                  0     1     1     1     1
United Arab Emirates     1     1     1     1     1


## 7. Indicator discovery and coverage
Technical codes remain alongside readable labels so later users can trace each measure to the workbook. Check availability before finalizing the historical and 2024 indicator groups. Missing observations represent unavailable data, never zero.

In [13]:
key_indicators = [
    "account.t.d",
    "fiaccount.t.d",
    "mobileaccount.t.d",
    "g20.any",
    "g20.made",
    "merchant.pay",
    "dig.acc",
    "save.any.t.d",
    "fin17a.17a1.d",
    "borrow.any.t.d",
    "fin22a.22a1.22g.d",
    "fin24aP",
    "internet",
    "con1",
    "con9a"
]

availability_2024 = (
    project_data[
        (project_data["year"] == 2024) &
        (project_data["group"] == "all")
    ][["countrynewwb"] + key_indicators]
    .set_index("countrynewwb")
    .notna()
)

availability_2024

Out[0]: 
                      account.t.d  fiaccount.t.d  ...  con1  con9a
countrynewwb                                      ...             
Algeria                      True           True  ...  True   True
Egypt, Arab Rep.             True           True  ...  True   True
Jordan                       True           True  ...  True   True
Morocco                      True           True  ...  True   True
Saudi Arabia                 True           True  ...  True   True
Tunisia                      True           True  ...  True   True
United Arab Emirates         True           True  ...  True   True

[7 rows x 15 columns]


In [14]:
availability_2024.sum(axis=1).sort_values(ascending=False)

Out[0]: 
countrynewwb
Egypt, Arab Rep.        15
Jordan                  15
Morocco                 15
Tunisia                 15
Saudi Arabia            14
Algeria                 13
United Arab Emirates     7
dtype: int64


In [15]:
historical_availability = (
    project_data[
        project_data["group"] == "all"
    ][["countrynewwb", "year"] + key_indicators]
    .set_index(["countrynewwb", "year"])
    .notna()
)

historical_counts = historical_availability.sum(axis=1).unstack()

historical_counts

Out[0]: 
year                  2011  2014  2017  2021  2024
countrynewwb                                      
Algeria                3.0   9.0   9.0  10.0  13.0
Egypt, Arab Rep.       3.0  10.0  10.0  11.0  15.0
Jordan                 3.0  10.0  10.0  11.0  15.0
Morocco                NaN   NaN  10.0  11.0  15.0
Saudi Arabia           3.0   9.0   9.0   9.0  14.0
Tunisia                NaN  10.0  10.0  11.0  15.0
United Arab Emirates   3.0  10.0  10.0  10.0   7.0


In [16]:
main_countries = [
    "Egypt, Arab Rep.",
    "Morocco",
    "Tunisia",
    "Algeria",
    "Jordan",
    "Saudi Arabia"
]

In [17]:
main_data = project_data[
    project_data["countrynewwb"].isin(main_countries)
].copy()

In [18]:
overall_main = main_data[
    main_data["group"] == "all"
]

In [19]:
indicator_coverage = {}

for year in sorted(overall_main["year"].unique()):
    year_data = overall_main[overall_main["year"] == year]

    indicator_coverage[year] = year_data[key_indicators].notna().sum()

In [20]:
indicator_coverage_df = pd.DataFrame(indicator_coverage)

indicator_coverage_df

Out[0]: 
                   2011  2014  2017  2021  2024
account.t.d           4     5     6     6     6
fiaccount.t.d         4     5     6     6     6
mobileaccount.t.d     0     3     4     4     5
g20.any               0     5     6     6     6
g20.made              0     5     6     6     4
merchant.pay          0     0     0     5     6
dig.acc               0     0     0     0     6
save.any.t.d          0     5     6     6     6
fin17a.17a1.d         4     5     6     6     6
borrow.any.t.d        0     5     6     6     6
fin22a.22a1.22g.d     0     5     6     6     6
fin24aP               0     5     6     6     6
internet              0     0     0     0     6
con1                  0     0     0     0     6
con9a                 0     0     0     0     6


In [21]:
historical_indicators = [
    "account.t.d",
    "fiaccount.t.d",
    "g20.any",
    "save.any.t.d",
    "fin17a.17a1.d",
    "borrow.any.t.d",
    "fin22a.22a1.22g.d",
    "fin24aP"
]

deep_dive_2024_indicators = [
    "mobileaccount.t.d",
    "merchant.pay",
    "dig.acc",
    "internet",
    "con1",
    "con9a"
]

In [22]:
print("Historical indicators:", len(historical_indicators))
print("2024 deep-dive indicators:", len(deep_dive_2024_indicators))

Historical indicators: 8
2024 deep-dive indicators: 6


In [23]:
demographic_2024 = main_data[
    main_data["year"] == 2024
]

demographic_coverage = (
    demographic_2024
    .groupby(["group", "group2"])[historical_indicators + deep_dive_2024_indicators]
    .count()
)

demographic_coverage

Out[0]: 
                                  account.t.d  fiaccount.t.d  ...  con1  con9a
group      group2                                             ...             
age_cat    age 25+                          6              6  ...     6      6
           ages 15-24                       6              6  ...     6      6
all        all                              6              6  ...     6      6
education  prim edu or less                 6              6  ...     6      6
           secondary edu or more            6              6  ...     6      6
gender     men                              6              6  ...     6      6
           women                            6              6  ...     6      6
income     poorest 40%                      6              6  ...     6      6
           richest 60%                      6              6  ...     6      6
laborforce in laborforce                    6              6  ...     6      6
           out of laborforce               

In [24]:
problem_indicators = [
    "fin17a.17a1.d",
    "fin22a.22a1.22g.d",
    "mobileaccount.t.d",
    "merchant.pay"
]

coverage_by_country_2024 = (
    main_data[
        (main_data["year"] == 2024) &
        (main_data["group"] == "all")
    ][["countrynewwb"] + problem_indicators]
    .set_index("countrynewwb")
    .notna()
)

coverage_by_country_2024

Out[0]: 
                  fin17a.17a1.d  ...  merchant.pay
countrynewwb                     ...              
Algeria                    True  ...          True
Egypt, Arab Rep.           True  ...          True
Jordan                     True  ...          True
Morocco                    True  ...          True
Saudi Arabia               True  ...          True
Tunisia                    True  ...          True

[6 rows x 4 columns]


In [25]:
demographic_groups = [
    "gender",
    "income",
    "age_cat",
    "urbanicity",
    "education",
    "laborforce"
]

core_demographic_indicators = [
    "account.t.d",
    "fiaccount.t.d",
    "g20.any",
    "save.any.t.d",
    "borrow.any.t.d",
    "fin24aP",
    "dig.acc",
    "internet",
    "con1",
    "con9a"
]

In [26]:
demographic_summary = {}

for group_name in demographic_groups:
    group_data = main_data[
        (main_data["year"] == 2024) &
        (main_data["group"] == group_name)
    ]

    demographic_summary[group_name] = (
        group_data[core_demographic_indicators]
        .notna()
        .sum()
    )

demographic_summary_df = pd.DataFrame(demographic_summary)

demographic_summary_df

Out[0]: 
                gender  income  age_cat  urbanicity  education  laborforce
account.t.d         12      12       12          12         12          12
fiaccount.t.d       12      12       12          12         12          12
g20.any             12      12       12          12         12          12
save.any.t.d        12      12       12          12         12          12
borrow.any.t.d      12      12       12          12         12          12
fin24aP             12      12       12          12         12          12
dig.acc             12      12       12          12         12          12
internet            12      12       12          12         12          12
con1                12      12       12          12         12          12
con9a               12      12       12          12         12          12


## 8. Focused analytical dataset and quality checks
Select the existing indicator framework and necessary source identifiers. Inspect uniqueness, missingness, and the expected proportion range before reshaping.

In [27]:
id_columns = [
    "countrynewwb",
    "codewb",
    "year",
    "pop_adult",
    "regionwb24_hi",
    "incomegroupwb24",
    "group",
    "group2"
]

In [28]:
selected_indicators = list(
    dict.fromkeys(
        historical_indicators + deep_dive_2024_indicators
    )
)

In [29]:
analysis_data = project_data[
    id_columns + selected_indicators
].copy()

In [30]:
print("Shape:", analysis_data.shape)
print("Number of indicators:", len(selected_indicators))

analysis_data.head()

Shape: (356, 22)
Number of indicators: 14
Out[0]: 
             countrynewwb codewb  year  ...  internet con1 con9a
2                 Algeria    DZA  2011  ...       NaN  NaN   NaN
40       Egypt, Arab Rep.    EGY  2011  ...       NaN  NaN   NaN
66                 Jordan    JOR  2011  ...       NaN  NaN   NaN
110          Saudi Arabia    SAU  2011  ...       NaN  NaN   NaN
133  United Arab Emirates    ARE  2011  ...       NaN  NaN   NaN

[5 rows x 22 columns]


In [31]:
duplicate_rows = analysis_data.duplicated(
    subset=[
        "countrynewwb",
        "year",
        "group",
        "group2"
    ]
)

print("Number of duplicate rows:", duplicate_rows.sum())

Number of duplicate rows: 0


In [32]:
missing_summary = pd.DataFrame({
    "missing_count": analysis_data.isna().sum(),
    "missing_percent": analysis_data.isna().mean() * 100
}).sort_values("missing_percent", ascending=False)

missing_summary

Out[0]: 
                   missing_count  missing_percent
merchant.pay                 311        87.359551
dig.acc                      278        78.089888
internet                     265        74.438202
con1                         265        74.438202
con9a                        265        74.438202
mobileaccount.t.d            248        69.662921
fin22a.22a1.22g.d            184        51.685393
fin17a.17a1.d                115        32.303371
g20.any                       68        19.101124
borrow.any.t.d                58        16.292135
save.any.t.d                  58        16.292135
fin24aP                       57        16.011236
codewb                         0         0.000000
countrynewwb                   0         0.000000
incomegroupwb24                0         0.000000
regionwb24_hi                  0         0.000000
pop_adult                      0         0.000000
year                           0         0.000000
group                          0         

In [33]:
missing_by_year = (
    analysis_data
    .groupby("year")[selected_indicators]
    .apply(lambda x: x.isna().mean() * 100)
)

missing_by_year.round(1)

Out[0]: 
      account.t.d  fiaccount.t.d  g20.any  ...  internet   con1  con9a
year                                       ...                        
2011          0.0            0.0    100.0  ...     100.0  100.0  100.0
2014          0.0            0.0     15.2  ...     100.0  100.0  100.0
2017          0.0            0.0      0.0  ...     100.0  100.0  100.0
2021          0.0            0.0      0.0  ...     100.0  100.0  100.0
2024          0.0            0.0     14.3  ...       0.0    0.0    0.0

[5 rows x 14 columns]


In [34]:
range_check = pd.DataFrame({
    "min": analysis_data[selected_indicators].min(),
    "max": analysis_data[selected_indicators].max()
})

range_check

Out[0]: 
                        min       max
account.t.d        0.054578  0.948826
fiaccount.t.d      0.054578  0.944342
g20.any            0.048772  0.923858
save.any.t.d       0.114119  0.725406
fin17a.17a1.d      0.000000  0.463634
borrow.any.t.d     0.158894  0.758910
fin22a.22a1.22g.d  0.013666  0.546740
fin24aP            0.000000  1.000000
mobileaccount.t.d  0.004687  0.523005
merchant.pay       0.030055  0.710353
dig.acc            0.028120  0.811267
internet           0.309153  1.000000
con1               0.772849  1.000000
con9a              0.339955  0.960758


In [35]:
invalid_values = (
    (analysis_data[selected_indicators] < 0) |
    (analysis_data[selected_indicators] > 1)
).sum()

invalid_values

Out[0]: 
account.t.d          0
fiaccount.t.d        0
g20.any              0
save.any.t.d         0
fin17a.17a1.d        0
borrow.any.t.d       0
fin22a.22a1.22g.d    0
fin24aP              0
mobileaccount.t.d    0
merchant.pay         0
dig.acc              0
internet             0
con1                 0
con9a                0
dtype: int64


## 9. Reshape to long format
One observation per country, year, demographic subgroup, and indicator supports later filtering and visual analysis.

In [36]:
analysis_long = analysis_data.melt(
    id_vars=id_columns,
    value_vars=selected_indicators,
    var_name="indicator",
    value_name="value"
)

print("Wide shape:", analysis_data.shape)
print("Long shape:", analysis_long.shape)

analysis_long.head(10)

Wide shape: (356, 22)
Long shape: (4984, 10)
Out[0]: 
           countrynewwb codewb  year  ...  group2    indicator     value
0               Algeria    DZA  2011  ...     all  account.t.d  0.332861
1      Egypt, Arab Rep.    EGY  2011  ...     all  account.t.d  0.097200
2                Jordan    JOR  2011  ...     all  account.t.d  0.254711
3          Saudi Arabia    SAU  2011  ...     all  account.t.d  0.464207
4  United Arab Emirates    ARE  2011  ...     all  account.t.d  0.597324
5               Algeria    DZA  2014  ...     all  account.t.d  0.504758
6      Egypt, Arab Rep.    EGY  2014  ...     all  account.t.d  0.141268
7                Jordan    JOR  2014  ...     all  account.t.d  0.246159
8          Saudi Arabia    SAU  2014  ...     all  account.t.d  0.694096
9               Tunisia    TUN  2014  ...     all  account.t.d  0.274323

[10 rows x 10 columns]


## 10. Readable indicator labels and cleaning
Keep the source code as `indicator_code`, attach a readable label, and exclude only observations with missing values. Newer indicators naturally have more historical gaps.

In [37]:
indicator_names = {
    "account.t.d": "Account ownership",
    "fiaccount.t.d": "Financial institution account",
    "g20.any": "Made or received digital payment",
    "save.any.t.d": "Saved any money",
    "fin17a.17a1.d": "Saved formally",
    "borrow.any.t.d": "Borrowed any money",
    "fin22a.22a1.22g.d": "Borrowed formally",
    "fin24aP": "Able to raise emergency funds",
    "mobileaccount.t.d": "Mobile money account",
    "merchant.pay": "Digital merchant payment",
    "dig.acc": "Digitally enabled account",
    "internet": "Internet use",
    "con1": "Mobile phone ownership",
    "con9a": "Smartphone ownership"
}

In [38]:
analysis_long["indicator_name"] = (
    analysis_long["indicator"].map(indicator_names)
)

analysis_long[
    ["indicator", "indicator_name"]
].drop_duplicates()

Out[0]: 
              indicator                    indicator_name
0           account.t.d                 Account ownership
356       fiaccount.t.d     Financial institution account
712             g20.any  Made or received digital payment
1068       save.any.t.d                   Saved any money
1424      fin17a.17a1.d                    Saved formally
1780     borrow.any.t.d                Borrowed any money
2136  fin22a.22a1.22g.d                 Borrowed formally
2492            fin24aP     Able to raise emergency funds
2848  mobileaccount.t.d              Mobile money account
3204       merchant.pay          Digital merchant payment
3560            dig.acc         Digitally enabled account
3916           internet                      Internet use
4272               con1            Mobile phone ownership
4628              con9a              Smartphone ownership


In [39]:
analysis_long_clean = analysis_long.dropna(
    subset=["value"]
).copy()

print("Before:", analysis_long.shape)
print("After:", analysis_long_clean.shape)

Before: (4984, 11)
After: (2812, 11)


In [40]:
analysis_long_clean = analysis_long_clean.rename(columns={
    "countrynewwb": "country",
    "codewb": "country_code",
    "pop_adult": "adult_population",
    "regionwb24_hi": "region",
    "incomegroupwb24": "income_group",
    "group": "demographic_group",
    "group2": "demographic_subgroup",
    "indicator": "indicator_code"
})
analysis_long_clean.columns

Out[0]: 
Index(['country', 'country_code', 'year', 'adult_population', 'region',
       'income_group', 'demographic_group', 'demographic_subgroup',
       'indicator_code', 'value', 'indicator_name'],
      dtype='object')


In [41]:
analysis_long_clean["country"] = analysis_long_clean["country"].replace({
    "Egypt, Arab Rep.": "Egypt"
})

## 11. Country roles
Assign the main market, peers, advanced benchmark, and supplementary benchmark explicitly. UAE is supplementary because detailed indicator coverage is weaker.

In [42]:
country_roles = {
    "Egypt": "Main market",
    "Morocco": "Peer market",
    "Tunisia": "Peer market",
    "Algeria": "Peer market",
    "Jordan": "Peer market",
    "Saudi Arabia": "Advanced benchmark",
    "United Arab Emirates": "Supplementary benchmark"
}

analysis_long_clean["country_role"] = (
    analysis_long_clean["country"].map(country_roles)
)

In [43]:
analysis_long_clean[
    ["country", "country_role"]
].drop_duplicates().sort_values("country")

Out[0]: 
                 country             country_role
0                Algeria              Peer market
1                  Egypt              Main market
2                 Jordan              Peer market
14               Morocco              Peer market
3           Saudi Arabia       Advanced benchmark
9                Tunisia              Peer market
4   United Arab Emirates  Supplementary benchmark


## 12. Final validation
Confirm the expected output schema, country and indicator coverage, complete retained observations, uniqueness, and proportion bounds.

In [44]:
print("Rows:", analysis_long_clean.shape[0])
print("Columns:", analysis_long_clean.shape[1])

print("\nMissing values:")
print(analysis_long_clean.isna().sum())

print("\nDuplicate rows:")
print(
    analysis_long_clean.duplicated(
        subset=[
            "country",
            "year",
            "demographic_group",
            "demographic_subgroup",
            "indicator_code"
        ]
    ).sum()
)

print("\nValue range:")
print(
    analysis_long_clean["value"].min(),
    analysis_long_clean["value"].max()
)

Rows: 2812
Columns: 12

Missing values:
country                 0
country_code            0
year                    0
adult_population        0
region                  0
income_group            0
demographic_group       0
demographic_subgroup    0
indicator_code          0
value                   0
indicator_name          0
country_role            0
dtype: int64

Duplicate rows:
0

Value range:
0.0 1.0


In [45]:
expected_columns = [
    "country", "country_code", "year", "adult_population", "region",
    "income_group", "demographic_group", "demographic_subgroup",
    "indicator_code", "value", "indicator_name", "country_role"
]
observation_key = ["country", "year", "demographic_group",
                   "demographic_subgroup", "indicator_code"]
assert analysis_long_clean.shape == (2812, 12)
assert analysis_long_clean.columns.tolist() == expected_columns
assert set(analysis_long_clean["country"]) == set(country_roles)
assert set(analysis_long_clean["indicator_code"]) == set(selected_indicators)
assert not analysis_long_clean.isna().any().any()
assert not analysis_long_clean.duplicated(subset=observation_key).any()
assert analysis_long_clean["value"].between(0, 1).all()
print("Validation passed: 2,812 observations, 12 columns, seven countries, 14 indicators.")

Validation passed: 2,812 observations, 12 columns, seven countries, 14 indicators.


## 13. Export and read-back verification
Save the analytical table using a project-relative path, then reload it to check that the exported file preserves the expected structure and values.

In [46]:
processed_file_path = "../data/processed/financial_inclusion_long_clean.csv"

analysis_long_clean.to_csv(
    processed_file_path,
    index=False
)

print("Saved successfully:", processed_file_path)

Saved successfully: ../data/processed/financial_inclusion_long_clean.csv


In [47]:
processed_check = pd.read_csv(processed_file_path)

print("Saved shape:", processed_check.shape)
processed_check.head()

Saved shape: (2812, 12)
Out[0]: 
                country  ...             country_role
0               Algeria  ...              Peer market
1                 Egypt  ...              Main market
2                Jordan  ...              Peer market
3          Saudi Arabia  ...       Advanced benchmark
4  United Arab Emirates  ...  Supplementary benchmark

[5 rows x 12 columns]


In [48]:
assert processed_check.shape == analysis_long_clean.shape
assert processed_check.columns.tolist() == expected_columns
pd.testing.assert_frame_equal(processed_check, analysis_long_clean.reset_index(drop=True),
                              check_dtype=False, check_exact=False, rtol=1e-12)
print("Export read-back verified.")

Export read-back verified.


## 14. Output summary
The processed Findex table contains 2,812 observed indicator values in 12 columns for seven project countries. The strongest common historical comparison is 2017–2024; 2024 offers the richer deep dive. Missing source observations were excluded from the final long table without imputation. The notebook stops at Findex preparation.